In [1]:
import pandas as pd

In [2]:
import pickle

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [4]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("nyc-taxi-experiment")

2026/03/30 16:39:32 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///C:/Users/AvadhootHublikar/Documents/mlops/mlops-zoomcamp/03-model_training/experiment_tracking/homework/artifacts_local/4', creation_time=1774868972272, experiment_id='4', last_update_time=1774868972272, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, workspace='default'>

In [5]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    # df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    # df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] =  df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60 )

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

                   
    return df

In [6]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [7]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)


In [8]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [ ]:
import xgboost as xgb

In [13]:
from pathlib import Path

models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [14]:
with mlflow.start_run():

    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.095,
        'max_depth': 30,
        'min_child_weight': 1.060,
        'objective': 'reg:linear',
        'reg_alpha': 0.018,
        'reg_lambda': 0.011,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=30,
        evals=[(valid, 'validation')],
        early_stopping_rounds=20
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric('rmse', rmse)


    with open('models/preprocessor.b', 'wb') as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact('models/preprocessor.b', artifact_path='preprocessor')

    mlflow.xgboost.log_model(booster, artifact_path='models_mlflow')  

c:\Users\AvadhootHublikar\Documents\mlops\exp-tracking\Lib\site-packages\xgboost\callback.py:385: UserWarning: [16:41:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\objective\regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.43283
[1]	validation-rmse:10.74812
[2]	validation-rmse:10.14789
[3]	validation-rmse:9.62334
[4]	validation-rmse:9.16409
[5]	validation-rmse:8.76892
[6]	validation-rmse:8.42824
[7]	validation-rmse:8.13172
[8]	validation-rmse:7.87891
[9]	validation-rmse:7.66113
[10]	validation-rmse:7.47243
[11]	validation-rmse:7.31143
[12]	validation-rmse:7.17291
[13]	validation-rmse:7.05274
[14]	validation-rmse:6.95364
[15]	validation-rmse:6.86626
[16]	validation-rmse:6.79175
[17]	validation-rmse:6.72681
[18]	validation-rmse:6.67122
[19]	validation-rmse:6.62314
[20]	validation-rmse:6.57967
[21]	validation-rmse:6.54338
[22]	validation-rmse:6.51309
[23]	validation-rmse:6.48445
[24]	validation-rmse:6.46066
[25]	validation-rmse:6.44015
[26]	validation-rmse:6.42200
[27]	validation-rmse:6.40572
[28]	validation-rmse:6.39050
[29]	validation-rmse:6.37695


2026/03/30 16:41:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run flawless-goat-987 at: http://127.0.0.1:5000/#/experiments/4/runs/de627bf3185343799a7916d0b66086e3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
